# Model Training

Prepare the Training Dataset and experiment with different models for automatically predict the `Test Results` based on a list of patient's features.

# Setup Notebook

## Imports

In [1]:
# Import Standard Libraries
import os
import mlflow
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# Import Package Modules
from src.general_utils.general_utils import read_configuration
from src.data_preparation.data_preparation import HealthcareDataPreparation
from src.model_training.model_training import ModelTrainer

## Define Configurations

In [2]:
# Retrieve root path
root_path = Path(os.getcwd()).parents[1]

# Read configuration variables
config = read_configuration(root_path / 
                            'configuration' / 
                            'healthcare_classification_config.yaml')

# Extract configuration variables
dataset_config = config['healthcare_dataset_config']
data_pipeline_config = config['healthcare_data_pipeline_config']
model_training_config = config['model_training_config']

[07/30/2024 14:12:55 - general_utils] INFO - read_configuration - Start
[07/30/2024 14:12:55 - general_utils] INFO - read_configuration - Reading /Users/s.porreca/Projects/MediBioticsAI/configuration/healthcare_classification_config.yaml
[07/30/2024 14:12:55 - general_utils] INFO - read_configuration - Configuration file /Users/s.porreca/Projects/MediBioticsAI/configuration/healthcare_classification_config.yaml read successfully
[07/30/2024 14:12:55 - general_utils] INFO - read_configuration - End


# Read Data

## Healthcare Dataset

In [3]:
# Read data
data = pd.read_csv((root_path / '/'.join(dataset_config['data_path'])),
                   parse_dates=dataset_config['date_columns'])

# Data Pipeline

## Define Features and Labels

In [4]:
# Define the features to include
features = data_pipeline_config['features']['numerical'] + \
           data_pipeline_config['features']['categorical']

# Define the label to include
label = data_pipeline_config['labels']

print('Features:')
[print(f'{index + 1}. {feature}') for index, feature in enumerate(features)]
print()
print(f'Labels: {label}')

Features:
1. Age
2. Billing Amount
3. Room Number
4. Gender
5. Blood Type
6. Medical Condition
7. Insurance Provider
8. Admission Type
9. Medication

Labels: ['Test Results']


## Define Data Preparation Pipeline

In [5]:
# Instance the data preparation pipeline object
data_preparation = HealthcareDataPreparation(data_pipeline_config['data_transformations'],
                                             data_pipeline_config['features'])

[07/30/2024 14:12:55 - HealthcareDataPreparation] INFO - __init__ - Initialise object attributes


In [6]:
# Get the training data preparation pipeline
data_preparation_pipeline = data_preparation.build_training_data_preparation_pipeline()

[07/30/2024 14:12:55 - HealthcareDataPreparation] INFO - build_training_data_preparation_pipeline - Start
[07/30/2024 14:12:55 - HealthcareDataPreparation] INFO - build_training_data_preparation_pipeline - Build the Numerical Data Pipeline
[07/30/2024 14:12:55 - data_preparation_utils] INFO - build_numerical_data_pipeline_steps - Start
[07/30/2024 14:12:55 - data_preparation_utils] INFO - build_numerical_data_pipeline_steps - Building steps
[07/30/2024 14:12:55 - data_preparation_utils] INFO - build_numerical_data_pipeline_steps - Skipping Feature Engineering step
[07/30/2024 14:12:55 - data_preparation_utils] INFO - build_numerical_data_pipeline_steps - Adding SimpleImputer Imputation step
[07/30/2024 14:12:55 - data_preparation_utils] INFO - build_numerical_data_pipeline_steps - Adding MinMaxScaler Standardisation step
[07/30/2024 14:12:55 - data_preparation_utils] INFO - build_numerical_data_pipeline_steps - Skipping Normalization step
[07/30/2024 14:12:55 - data_preparation_utils] 

In [7]:
data_preparation_pipeline

ColumnTransformer(transformers=[('numerical',
                                 Pipeline(steps=[('imputation',
                                                  SimpleImputer(copy=False,
                                                                strategy='median')),
                                                 ('standardisation',
                                                  MinMaxScaler())]),
                                 ['Age', 'Billing Amount', 'Room Number']),
                                ('categorical',
                                 Pipeline(steps=[('imputation',
                                                  SimpleImputer(copy=False,
                                                                fill_value='unknown',
                                                                strategy='constant')),
                                                 ('one_hot_encoding',
                                                  OneHotEncoder())]),
                                 ['Gender', 'Blood Type', 'Medical Condition',
                                  'Insurance Provider', 'Admission Type',
                                  'Medication'])])

## Encode Label

In [8]:
# Encode the label
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(np.ravel(data[label]))

## Train & Test Split

In [9]:
# Define X and y for the training set
X = data[features]
y = encoded_labels

In [10]:
# Retrieve test_size and random_state
test_size = data_pipeline_config['train_test_split']['test_size']
random_state = data_pipeline_config['train_test_split']['random_state']

In [11]:
# Split training data into train and validation
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=test_size,
                                                    random_state=random_state)

# Model Training

## Setup Training

In [12]:
# Set MLflow Experiment
mlflow_experiment_name = model_training_config['mlflow']['experiment_name']

# Set MLflow Experiment
mlflow.set_experiment(mlflow_experiment_name)

<Experiment: artifact_location='file:///Users/s.porreca/Projects/MediBioticsAI/notebooks/healthcare_classification/mlruns/388746619317475161', creation_time=1714156779505, experiment_id='388746619317475161', last_update_time=1714156779505, lifecycle_stage='active', name='Version 1.0.x', tags={}>

In [13]:
# Initialise trained models dictionary
models = {}

# Initialize DataFrame of models performance
performance = pd.DataFrame(columns=model_training_config['metrics'])

## Logistic Regression

In [14]:
# Extract model training configurations
model_name = model_training_config['logistic_regression']['model_name']
mlflow_run_name = model_training_config['logistic_regression']['mlflow_run_name']
metrics = model_training_config['metrics']

In [15]:
# Define the model
model_lr = LogisticRegression()

# Create a ModelTrainer
model_trainer_lr = ModelTrainer(model_name=model_name, 
                                model=model_lr, 
                                data_pipeline=data_preparation_pipeline)

[07/30/2024 14:12:55 - ModelTrainer] INFO - __init__ - Initialise object attributes


In [16]:
# Start an MLflow run
with mlflow.start_run(run_name=mlflow_run_name):

    # Fit the model trainer
    model_trainer_lr.bundle_and_fit_pipeline(X_train, y_train)
    
    # Evaluate the model trainer
    evaluation = model_trainer_lr.evaluate_pipeline(X_test, y_test, metrics)
    
    # Log model's evaluation metrics
    mlflow.log_metrics(evaluation.to_dict()['Value'])
    
    # Log model's features
    mlflow.log_params({'Features': features, 
                       'Labels': label,
                       'Data Transformations': data_pipeline_config['data_transformations'],
                       'Model Initial Hyperparameters': None,
                       'Model Optimised Hyperparameters': None})
    
    # Log the model
    _ = mlflow.sklearn.log_model(
        sk_model=model_trainer_lr.pipeline,
        artifact_path=model_name,
        input_example=X_train.head(1),
        registered_model_name=mlflow_run_name
    )

[07/30/2024 14:12:55 - ModelTrainer] INFO - bundle_and_fit_pipeline - Start
[07/30/2024 14:12:55 - ModelTrainer] INFO - bundle_and_fit_pipeline - Bundle the pipeline
[07/30/2024 14:12:55 - ModelTrainer] INFO - bundle_and_fit_pipeline - Fit the pipeline
[07/30/2024 14:12:55 - ModelTrainer] INFO - bundle_and_fit_pipeline - End
[07/30/2024 14:12:55 - ModelTrainer] INFO - evaluate_pipeline - Start
[07/30/2024 14:12:55 - ModelTrainer] INFO - evaluate_pipeline - Compute predictions
[07/30/2024 14:12:55 - ModelTrainer] INFO - evaluate_pipeline - Evaluate pipeline
[07/30/2024 14:12:55 - model_training_utils] INFO - compute_multi_classification_metrics - Start
[07/30/2024 14:12:55 - model_training_utils] INFO - compute_multi_classification_metrics - Compute metrics            Value
Accuracy    0.32
Precision   0.32
Recall      0.32
F1 Score    0.32
ROC AUC     0.49
[07/30/2024 14:12:55 - model_training_utils] INFO - compute_multi_classification_metrics - End
[07/30/2024 14:12:55 - ModelTrainer]

/Users/s.porreca/Projects/MediBioticsAI/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/s.porreca/Projects/MediBioticsAI/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missi

In [17]:
# Update performance dataframe for Model Explainability
performance.loc[mlflow_run_name] = evaluation.Value.values

# Update models dataframe for Model Explainability
models[model_name] = model_trainer_lr.pipeline

## XGBoost Classifier

In [18]:
# Extract model training configurations
model_name = model_training_config['xgboost']['model_name']
mlflow_run_name = model_training_config['xgboost']['mlflow_run_name']
parameters = model_training_config['xgboost']['parameters']
metrics = model_training_config['metrics']

In [19]:
# Define the model
model_xgb = XGBClassifier(**parameters)

# Create a ModelTrainer
model_trainer_xgb = ModelTrainer(model_name=model_name, 
                                 model=model_xgb, 
                                 data_pipeline=data_preparation_pipeline)

[07/30/2024 14:12:59 - ModelTrainer] INFO - __init__ - Initialise object attributes


In [20]:
# Start an MLflow run
with mlflow.start_run(run_name=mlflow_run_name):

    # Fit the model trainer
    model_trainer_xgb.bundle_and_fit_pipeline(X_train, y_train)
    
    # Evaluate the model trainer
    evaluation = model_trainer_xgb.evaluate_pipeline(X_test, y_test, metrics)
    
    # Log model's evaluation metrics
    mlflow.log_metrics(evaluation.to_dict()['Value'])
    
    # Log model's features
    mlflow.log_params({'Features': features, 
                       'Labels': label,
                       'Data Transformations': data_pipeline_config['data_transformations'],
                       'Model Initial Hyperparameters': None,
                       'Model Optimised Hyperparameters': None})
    
    # Log the model
    _ = mlflow.sklearn.log_model(
        sk_model=model_trainer_xgb.pipeline,
        artifact_path=model_name,
        input_example=X_train.head(1),
        registered_model_name=mlflow_run_name
    )

[07/30/2024 14:12:59 - ModelTrainer] INFO - bundle_and_fit_pipeline - Start
[07/30/2024 14:12:59 - ModelTrainer] INFO - bundle_and_fit_pipeline - Bundle the pipeline
[07/30/2024 14:12:59 - ModelTrainer] INFO - bundle_and_fit_pipeline - Fit the pipeline
[07/30/2024 14:13:00 - ModelTrainer] INFO - bundle_and_fit_pipeline - End
[07/30/2024 14:13:00 - ModelTrainer] INFO - evaluate_pipeline - Start
[07/30/2024 14:13:00 - ModelTrainer] INFO - evaluate_pipeline - Compute predictions
[07/30/2024 14:13:00 - ModelTrainer] INFO - evaluate_pipeline - Evaluate pipeline
[07/30/2024 14:13:00 - model_training_utils] INFO - compute_multi_classification_metrics - Start
[07/30/2024 14:13:00 - model_training_utils] INFO - compute_multi_classification_metrics - Compute metrics            Value
Accuracy    0.32
Precision   0.32
Recall      0.32
F1 Score    0.32
ROC AUC     0.50
[07/30/2024 14:13:00 - model_training_utils] INFO - compute_multi_classification_metrics - End
[07/30/2024 14:13:00 - ModelTrainer]

/Users/s.porreca/Projects/MediBioticsAI/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/s.porreca/Projects/MediBioticsAI/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missi

In [21]:
# Update performance dataframe for Model Explainability
performance.loc[mlflow_run_name] = evaluation.Value.values

# Update models dataframe for Model Explainability
models[model_name] = model_trainer_xgb.pipeline